# 03 · Sampling: from noise back to data

The payoff. Start from pure noise $x_T\sim\mathcal N(0,I)$ and reverse one step at
a time. Each DDPM reverse step uses the predicted noise to form the posterior
mean, then adds a little fresh noise (except on the final step):

$$x_{t-1}=\underbrace{\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\Big)}_{\text{mean}}+\sqrt{\tilde\beta_t}\,z,\qquad z\sim\mathcal N(0,I)$$

with posterior variance
$\tilde\beta_t=\beta_t\dfrac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}$.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each one is followed by a
> **self-check** cell — run it and it will tell you if your implementation is
> correct (it compares against the reference and asserts the key properties).
>
> **Stuck?** The answer key is `solutions/notebooks/` (with plots), and the
> reference implementation lives in the `nanodiffusion/` package. Peeking is
> allowed — but try first.

In [ ]:
import torch
import matplotlib.pyplot as plt

from nanodiffusion.utils import pick_device, set_seed, scatter_2d
from nanodiffusion.data import toy2d
from nanodiffusion.schedules import NoiseSchedule
from nanodiffusion.models import MLPDenoiser
from nanodiffusion.objectives import ddpm_eps_loss
from nanodiffusion.samplers import ddpm_sample as reference_sample   # for the self-check

set_seed(0)
device = pick_device()
schedule = NoiseSchedule.make("cosine", 200).to(device)
data = toy2d("swiss_roll", 8000).to(device)
print("device:", device)

## Get a trained model

We reuse the reference `MLPDenoiser` here so this notebook's TODO stays focused
purely on the **sampler**. (Trains in a few seconds.)

In [ ]:
set_seed(0)
model = MLPDenoiser().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(2000):
    idx = torch.randint(0, data.shape[0], (512,), device=device)
    loss = ddpm_eps_loss(model, data[idx], schedule)
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()
print(f"trained, final loss {loss.item():.4f}")

## TODO — the ancestral sampler

Fill in the reverse loop. Useful pieces already on `schedule` (all length-`T`
tensors, already on the right device):

`schedule.betas`, `schedule.alphas`, `schedule.alpha_bars`,
`schedule.alpha_bars_prev`, `schedule.sqrt_one_minus_alpha_bars`

For 2D data of shape `(B, 2)`, index with `[step]` (a plain int) — all these are
scalars per timestep, so no reshaping is needed.

In [ ]:
@torch.no_grad()
def my_ddpm_sample(model, schedule, shape, device, return_trajectory=False):
    '''Reverse the diffusion process: pure noise -> data.'''
    x = torch.randn(shape, device=device)        # start from x_T
    traj = [x.clone()]

    for step in reversed(range(schedule.timesteps)):
        t = torch.full((shape[0],), step, device=device, dtype=torch.long)
        eps = model(x, t)                         # predicted noise

        beta_t = schedule.betas[step]
        alpha_t = schedule.alphas[step]
        sqrt_1m_ab = schedule.sqrt_one_minus_alpha_bars[step]

        # TODO 1: the posterior mean
        #   mean = (x - beta_t / sqrt_1m_ab * eps) / sqrt(alpha_t)
        mean = ...  # <- replace

        if step > 0:
            # TODO 2: add fresh noise scaled by sqrt of the posterior variance
            #   posterior_var = beta_t * (1 - alpha_bars_prev[step]) / (1 - alpha_bars[step])
            #   x = mean + sqrt(posterior_var) * torch.randn_like(x)
            x = ...  # <- replace
        else:
            x = mean          # final step: no noise, return the clean estimate

        if return_trajectory:
            traj.append(x.clone())

    return (x, traj) if return_trajectory else x

In [ ]:
# ---- self-check ----
# Same seed + same model => your sampler should match the reference exactly.
set_seed(123); mine = my_ddpm_sample(model, schedule, (2000, 2), device)
set_seed(123); ref = reference_sample(model, schedule, (2000, 2), device=device)
assert mine.shape == (2000, 2), f"shape {tuple(mine.shape)} != (2000, 2)"
assert torch.isfinite(mine).all(), "produced NaN/inf — check the mean formula"
assert torch.allclose(mine, ref, atol=1e-4), "doesn't match the reference sampler"

# and the generated distribution should match the real one
real_std, gen_std = data.std(0).cpu(), mine.std(0).cpu()
print(f"real std {real_std.tolist()}\ngen  std {gen_std.tolist()}")
assert torch.allclose(gen_std, real_std, atol=0.25), "distribution doesn't match the data"
print("✅ sampler correct")

## Look at your samples

In [ ]:
gen = my_ddpm_sample(model, schedule, (2000, 2), device)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 4))
scatter_2d(a1, data[:2000], "real data")
scatter_2d(a2, gen, "YOUR generated samples", color="C1")
plt.tight_layout(); plt.show()

## Watch the reverse trajectory

The noise blob (left) reorganizing into the swiss roll (right) — notebook 00's
forward process, run backwards.

In [ ]:
_, traj = my_ddpm_sample(model, schedule, (2000, 2), device, return_trajectory=True)
picks = [0, 40, 80, 120, 160, 200]
fig, axes = plt.subplots(1, len(picks), figsize=(2.6 * len(picks), 2.6))
for ax, i in zip(axes, picks):
    scatter_2d(ax, traj[i], f"t = {max(schedule.timesteps - i, 0)}", color="C1")
plt.suptitle("Your reverse process: noise -> swiss roll")
plt.tight_layout(); plt.show()

## 🎉 You built a diffusion model — by hand

Forward noising, a time-conditioned ε-predictor, the MSE loss, and an ancestral
sampler. All yours.

Compare your code against `nanodiffusion/` and `solutions/notebooks/`.

**Part 2 next:** swap the MLP for a **U-Net** and point this same machinery at
MNIST for your first real images.